In [1]:
# Model agnostic 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
from typing import Optional, List, Callable, Dict, Any, List
from pathlib import Path
from src.utils import Helm  # custom model for data handling/model trianing

# Model specific 
from xgboost import XGBRegressor, XGBClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from typing import Optional, List 

from sklearn.linear_model import LinearRegression

This version of Feyn and the QLattice is available for academic, personal, and non-commercial use. By using the community version of this software you agree to the terms and conditions which can be found at https://abzu.ai/eula.

In [2]:
# Get the directory this file lives in
nb_dir = Path.cwd() # notebook directory
project_root = nb_dir.parents[0] # project directory
data_path = project_root / "datasets" / "processed_well_data.csv"

drop_cols = ['Dia', 'Dev(deg)','Area (m2)', 'z','GasDens','LiquidDens', 'P/T','friction_factor', 'critical_film_thickness', 'Test status', 'Qcr', 'Gasflowrate', 'ΔQ']
D = Helm(path=data_path, drop_cols=drop_cols, test_size=0.20)

In [3]:
# define xgboost multiclass pipeline
def xgboost(
        hparams: Dict[str, Any]
) -> Pipeline:
    
    xgb = XGBClassifier(
        objective="multi:softprob",   # multiclass with probabilities
        eval_metric="mlogloss",
        num_class=3,  # remove from hparams and pass explicitly
        random_state=42,
        importance_type="gain",
        **hparams,
    )

    selector = SelectFromModel(
        estimator=xgb,
        threshold="mean",
        prefit=False
    )

    pipe = Pipeline([
        ("feature_sel", selector),
        ("model", xgb),
    ])

    return pipe

hparam_grid = {
    "n_estimators":  [40],
    "learning_rate": [0.10],
    "max_depth":     [10],
}

trained_model = D.evolv_model(
    build_model=xgboost,
    hparam_grid=hparam_grid,
    k_folds=5
)

Training model and optimizing hyperparameters via k-fold CV...
Retraining optimized model on full training set...

Best CV Classification Accuracy: 
>>> 0.8672 ± 0.0316

Best CV Per-Class Accuracy:
>>> Unloaded (0): 0.8977, Near L.U. (1): 0.6000, Loaded (2): 0.8493

Best Hyperparameters:
>>> {'n_estimators': 40, 'learning_rate': 0.1, 'max_depth': 10}

Training Classification Accuracy:
>>> 1.0000

Per-Class Training Accuracy:
>>> Unloaded (0): 1.0000, Near L.U. (1): 1.0000, Loaded (2): 1.0000

Test Classification Accuracy:
>>> 0.8571

Per-Class Test Accuracy:
>>> Unloaded (0): 0.9130, Near L.U. (1): 1.0000, Loaded (2): 0.7778
